# Chatbot Example


In this lesson, you will familiarize yourself with the chatbot example you will work on during this chapter(Chapter 4). The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.


## Import Libraries


In [ ]:
!pip install openai arxiv python-dotenv -q

In [1]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv

## Tool Functions


In [2]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  


In [3]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    client = arxiv.Client()

    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    print(f"Saving papers information in: {path}")
    
    file_path = os.path.join(path, "papers_info.json")

    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids


In [4]:
search_papers("computers")


Saving papers information in: papers\computers
Results are saved in: papers\computers\papers_info.json


['1312.3300v1', '2207.05241v1', '2603.19778v1', '2601.11095v1', '2012.10468v1']

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.


In [5]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."


In [6]:
extract_info('1312.3300v1')


'{\n  "title": "Numerical Reproducibility and Parallel Computations: Issues for Interval Algorithms",\n  "authors": [\n    "Nathalie Revol",\n    "Philippe Th\\u00e9veny"\n  ],\n  "summary": "What is called \\"numerical reproducibility\\" is the problem of getting the same result when the scientific computation is run several times, either on the same machine or on different machines, with different types and numbers of processing units, execution environments, computational loads etc. This problem is especially stringent for HPC numerical simulations. In what follows, the focus is on parallel implementations of interval arithmetic using floating-point arithmetic. For interval computations, numerical reproducibility is of course an issue for testing and debugging purposes. However, as long as the computed result encloses the exact and unknown result, the inclusion property, which is the main property of interval arithmetic, is satisfied and getting bit for bit identical results may not

## Tool Schema


Here are the schemas of each tool in **OpenAI function-calling format**.


In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": (
                "Search arXiv for academic papers about a topic. "
                "Always use this tool when the user asks to find, "
                "search, discover, recommend, or list papers."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": (
                            "The academic topic to search on arXiv, "
                            "for example: supersymmetry at the LHC."
                        ),
                    },
                    "max_results": {
                        "type": "integer",
                        "description": (
                            "Maximum number of papers to return."
                        ),
                        "default": 5,
                    },
                },
                "required": ["topic"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "extract_info",
            "description": (
                "Retrieve saved metadata for one specific arXiv "
                "paper using its paper ID."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": (
                            "The arXiv paper ID, such as 1312.3300v1."
                        ),
                    }
                },
                "required": ["paper_id"],
                "additionalProperties": False,
            },
        },
    },
]

## Tool Mapping


This code handles tool mapping and execution.


In [9]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
    elif isinstance(result, list):
        result = ', '.join(result)
    elif isinstance(result, dict):
        result = json.dumps(result, indent=2)
    else:
        result = str(result)
    return result


## Chatbot Code


The chatbot handles the user's queries one by one, but it does not persist memory across the queries.


In [10]:
import aisuite as ai

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")

client = ai.Client(
    {
        "openai": {
            "api_key": openai_api_key,
            "base_url": openai_base_url,
        }
    }
)


### Query Processing


In [11]:
def process_query(query: str) -> None:
    messages = [
        {
            "role": "system",
            "content": """
You are a research-paper assistant.

Rules:
- When the user asks to find, search for, discover, or list papers
  about a subject, you MUST call the search_papers tool.
- When the user asks for information about a specific paper ID,
  you MUST call the extract_info tool.
- Do not invent paper titles, IDs, authors, or metadata.
- Use the tool results to answer the user.
""".strip(),
        },
        {
            "role": "user",
            "content": query,
        },
    ]

    while True:
        response = client.chat.completions.create(
            model="openai:gpt-5-nano",
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        message = response.choices[0].message

        # No tool request: final textual answer
        if not getattr(message, "tool_calls", None):
            print(message.content)
            break

        # Preserve the assistant tool-call message
        messages.append(message)

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name

            try:
                tool_args = json.loads(
                    tool_call.function.arguments
                )
            except json.JSONDecodeError as error:
                tool_result = f"Invalid tool arguments: {error}"

            else:
                print(
                    f"Calling tool {tool_name} "
                    f"with args {tool_args}"
                )

                try:
                    tool_result = execute_tool(
                        tool_name,
                        tool_args,
                    )
                except Exception as error:
                    tool_result = (
                        f"Tool execution failed: {error}"
                    )

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result),
                }
            )

### Chat Loop


In [12]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")


Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"


In [13]:
chat_loop()


Type your queries or 'quit' to exit.

Error: An error occurred: Error code: 400 - {'error': {'message': "Invalid value for 'tool_choice': 'tool_choice' is only allowed when 'tools' are specified. (request id: 20260804113520617019813PUiE8fGC)", 'type': 'v_api_biz_error', 'param': 'tool_choice', 'code': None}}


In [14]:
# ============================================================================
# CHATBOT WITH ARXIV PAPER SEARCH TOOLS
# ============================================================================
# Run this entire cell to start the chatbot with all functions defined.
# Type your queries about papers and the bot will search arXiv for you.
# ============================================================================

# --- Install dependencies (if needed) ---
!pip install openai arxiv python-dotenv aisuite -q

# --- Imports ---
import os
import json
import arxiv
from typing import List
from dotenv import load_dotenv
import aisuite as ai

# --- Constants ---
PAPER_DIR = "papers"

# --- Load Environment Variables ---
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("BASE_URL")

if not openai_api_key or not openai_base_url:
    raise ValueError("Missing OPENAI_API_KEY or BASE_URL in environment variables.")

# --- Initialize AI Client ---
client = ai.Client({
    "openai": {
        "api_key": openai_api_key,
        "base_url": openai_base_url,
    }
})

# ============================================================================
# TOOL FUNCTIONS
# ============================================================================

def search_papers(topic: str, max_results: int = 5) -> str:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        Formatted string with paper summaries and IDs
    """
    client_arxiv = arxiv.Client()
    
    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    
    papers = client_arxiv.results(search)
    
    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")
    
    # Load existing data (if any)
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}
    
    paper_list = []
    new_papers = []
    
    for paper in papers:
        paper_id = paper.get_short_id()
        paper_list.append(paper_id)
        
        # Only add if not already saved
        if paper_id not in papers_info:
            paper_info = {
                'title': paper.title,
                'authors': [author.name for author in paper.authors],
                'summary': paper.summary,
                'pdf_url': paper.pdf_url,
                'published': str(paper.published.date())
            }
            papers_info[paper_id] = paper_info
            new_papers.append(paper_id)
    
    # Save to JSON
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    # Build formatted response
    result = f"🔍 **Found {len(paper_list)} papers on '{topic}'**\n"
    result += f"📁 Saved in: {file_path}\n\n"
    
    if not paper_list:
        return "No papers found for this topic."
    
    for i, paper_id in enumerate(paper_list, 1):
        info = papers_info[paper_id]
        result += f"**{i}. {info['title']}**\n"
        result += f"   📅 {info['published']} | 👤 {', '.join(info['authors'][:3])}"
        if len(info['authors']) > 3:
            result += f" et al."
        result += "\n"
        result += f"   📄 ID: `{paper_id}`\n"
        result += f"   🔗 PDF: {info['pdf_url']}\n"
        result += f"   📝 Summary: {info['summary'][:200]}...\n\n"
    
    # Add helpful tips
    result += "💡 **Tips:**\n"
    result += "   - Use a paper ID (e.g., `2301.12345`) to get full details\n"
    result += "   - Ask me to 'show details for paper [ID]' or 'summarize paper [ID]'\n"
    result += f"   - Found {len(new_papers)} new papers added to the database.\n"
    
    return result


def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        Formatted string with paper information or error message
    """
    # Clean the paper ID
    paper_id = paper_id.strip()
    
    # Search through all topic directories
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            info = papers_info[paper_id]
                            
                            result = f"📄 **{info['title']}**\n"
                            result += f"👤 **Authors:** {', '.join(info['authors'])}\n"
                            result += f"📅 **Published:** {info['published']}\n"
                            result += f"🔗 **PDF URL:** {info['pdf_url']}\n"
                            result += f"📝 **Abstract:**\n{info['summary']}\n"
                            
                            return result
                except (FileNotFoundError, json.JSONDecodeError):
                    continue
    
    return f"❌ No saved information found for paper `{paper_id}`. Try searching for papers on a topic first."


def get_all_saved_papers() -> str:
    """
    Get a list of all papers saved in the database across all topics.
    
    Returns:
        Formatted string with all saved papers
    """
    all_papers = {}
    
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        for paper_id, info in papers_info.items():
                            if paper_id not in all_papers:
                                all_papers[paper_id] = info
                except:
                    continue
    
    if not all_papers:
        return "No papers saved yet. Search for a topic first!"
    
    result = f"📚 **Total saved papers: {len(all_papers)}**\n\n"
    for i, (paper_id, info) in enumerate(list(all_papers.items())[:20], 1):
        result += f"{i}. **{info['title'][:80]}...**\n"
        result += f"   📄 ID: `{paper_id}` | 📅 {info['published']}\n"
    
    if len(all_papers) > 20:
        result += f"\n... and {len(all_papers) - 20} more papers.\n"
    
    return result


# ============================================================================
# TOOL SCHEMAS (for OpenAI function calling)
# ============================================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search for papers on arXiv based on a topic. Saves all paper metadata and returns a summary with paper IDs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search for (e.g., 'LLM interpretability', 'quantum computing')"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to retrieve (default: 5)",
                        "default": 5
                    }
                },
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_info",
            "description": "Get detailed information about a specific paper by its ID. Returns title, authors, date, PDF URL, and abstract.",
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_id": {
                        "type": "string",
                        "description": "The arXiv paper ID (e.g., '2402.12317v2')"
                    }
                },
                "required": ["paper_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_all_saved_papers",
            "description": "Get a list of all papers saved in the database across all topics.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]


# ============================================================================
# TOOL MAPPING & EXECUTION
# ============================================================================

mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info,
    "get_all_saved_papers": get_all_saved_papers
}


def execute_tool(tool_name: str, tool_args: dict) -> str:
    """Execute a tool and return the result as a string."""
    try:
        result = mapping_tool_function[tool_name](**tool_args)
        
        # Convert result to string if needed
        if result is None:
            return "The operation completed but didn't return any results."
        elif isinstance(result, list):
            return '\n'.join(str(item) for item in result)
        elif isinstance(result, dict):
            return json.dumps(result, indent=2)
        else:
            return str(result)
    except Exception as e:
        return f"❌ Error executing {tool_name}: {str(e)}"


# ============================================================================
# QUERY PROCESSING
# ============================================================================

def process_query(query: str, max_turns: int = 5) -> str:
    """
    Process a user query with tool calling support.
    
    Args:
        query: The user's query
        max_turns: Maximum number of tool call rounds
        
    Returns:
        The final response from the assistant
    """
    messages = [{'role': 'user', 'content': query}]
    
    try:
        response = client.chat.completions.create(
            model='openai:gpt-4o',
            tools=tools,
            messages=messages
        )
        
        for turn in range(max_turns):
            message = response.choices[0].message
            
            # If no tool calls, return the response
            if not message.tool_calls:
                return message.content
            
            # Add assistant message with tool calls to history
            messages.append(message)
            
            # Execute all tool calls
            for tool_call in message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                
                print(f"🔧 Calling {tool_name} with {tool_args}")
                result = execute_tool(tool_name, tool_args)
                
                # Add tool result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
            
            # Get next response from model
            response = client.chat.completions.create(
                model='openai:gpt-4o',
                tools=tools,
                messages=messages
            )
        
        # If we've exhausted max turns, return the last response
        return response.choices[0].message.content or "Max turns reached without final answer."
        
    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============================================================================
# CHAT LOOP
# ============================================================================

def chat_loop():
    """Interactive chat loop with the paper search assistant."""
    print("\n" + "="*70)
    print("📚 ARXIV PAPER SEARCH ASSISTANT")
    print("="*70)
    print("\nI can help you find and summarize academic papers on any topic.")
    print("\nExamples:")
    print("  - 'Search for 3 papers on LLM interpretability'")
    print("  - 'Find papers about quantum computing'")
    print("  - 'Show me details for paper 2402.12317v2'")
    print("  - 'List all saved papers'")
    print("\nType 'quit' or 'exit' to stop.\n")
    
    while True:
        try:
            query = input("👤 You: ").strip()
            
            if not query:
                continue
                
            if query.lower() in ['quit', 'exit', 'q']:
                print("\n👋 Goodbye! Happy researching!")
                break
            
            print("\n🤖 Assistant: Processing...\n")
            response = process_query(query)
            print(response)
            print("\n" + "-"*70 + "\n")
            
        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            print("\n" + "-"*70 + "\n")


# ============================================================================
# START THE CHATBOT
# ============================================================================

if __name__ == "__main__":
    chat_loop()


📚 ARXIV PAPER SEARCH ASSISTANT

I can help you find and summarize academic papers on any topic.

Examples:
  - 'Search for 3 papers on LLM interpretability'
  - 'Find papers about quantum computing'
  - 'Show me details for paper 2402.12317v2'
  - 'List all saved papers'

Type 'quit' or 'exit' to stop.




[notice] A new release of pip is available: 23.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip



🤖 Assistant: Processing...

Certainly! Here are three significant papers on the interpretability of large language models (LLMs):

1. **"Attention is All You Need" (2017) by Vaswani et al.**
   - This foundational paper introduces the Transformer model, which is the basis for many current LLMs. While the primary focus is on the architecture itself, the paper discusses the self-attention mechanism, which provides some level of interpretability by allowing us to visualize which parts of the input the model focuses on for its predictions. Understanding this mechanism is crucial for interpreting the decisions made by LLMs.

2. **"Investigating BERT's Knowledge of Language: Five Analysis Methods with NPIs" (2019) by Warstadt et al.**
   - In this paper, the authors investigate the interpretability of BERT, a widely-used LLM, by analyzing how it handles Negative Polarity Items (NPIs). The study uses various analysis techniques to interpret what BERT knows about linguistic structures. This r